In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

(a)

In [ ]:
signal_path = "data/signal_Bs2MuMu.txt"
background_path = "data/background_combinatorial.txt"
features = ["PT1", "PT2", "P1", "P2", "TotalPT", "VertexChisq", "Isolation", "MASS"]

signal = pd.read_csv(signal_path, sep=r"\s+", header=None, names=features)
background = pd.read_csv(background_path, sep=r"\s+", header=None, names=features)
signal = signal.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
background = background.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)

os.makedirs("plots", exist_ok=True)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, feat in zip(axes.flatten(), features):
    all_vals = np.concatenate([signal[feat].values, background[feat].values])
    bins = np.linspace(all_vals.min(), all_vals.max(), 50)
    ax.hist(background[feat], bins=bins, alpha=0.6, label="Background", density=True)
    ax.hist(signal[feat], bins=bins, alpha=0.6, label="Signal", density=True)
    ax.set_title(feat); ax.set_xlabel(feat); ax.set_ylabel("Density"); ax.legend()

plt.tight_layout()
plt.savefig("plots/all_features_overview.png", dpi=300, bbox_inches="tight")
plt.show()

All features can be used except the mass as we will use the mass distribution to fit a signal peak over the background. Using mss in the classifier would therefore bias the distribution 

(b)

In [ ]:
def fisher_score(sig_col, bkg_col):
    sig_col = np.asarray(sig_col, dtype=float)
    bkg_col = np.asarray(bkg_col, dtype=float)
    all_vals = np.concatenate([sig_col, bkg_col])
    mu_sig, mu_bkg = sig_col.mean(), bkg_col.mean()
    mu_all = all_vals.mean()
    sigma2 = all_vals.var()
    if sigma2 == 0:
        return 0.0
    return (len(sig_col) * (mu_sig - mu_all) ** 2 + len(bkg_col) * (mu_bkg - mu_all) ** 2) / sigma2

scores = [{"Feature": f, "FisherScore": fisher_score(signal[f], background[f])} for f in features]
fisher_df = pd.DataFrame(scores).sort_values("FisherScore", ascending=False).reset_index(drop=True)
fisher_df["Rank"] = np.arange(1, len(fisher_df) + 1)
fisher_df = fisher_df[["Rank", "Feature", "FisherScore"]]
display(fisher_df)
fisher_df.to_csv("fisher_scores.csv", index=False)

In [ ]:
top3_features = fisher_df.loc[fisher_df["Feature"] != "MASS", "Feature"].head(3).tolist()
print("top 3 features:", top3_features)


def get_thresholds(sig_vals, bkg_vals, n_points=15):
    all_vals = np.concatenate([sig_vals, bkg_vals])
    return np.unique(np.percentile(all_vals, np.linspace(2, 98, n_points)))


def refine_thresholds(sig_vals, bkg_vals, best_t, width_fraction=0.08, n_points=25):
    all_vals = np.concatenate([sig_vals, bkg_vals])
    vmin, vmax = all_vals.min(), all_vals.max()
    vrange = vmax - vmin
    low = max(vmin, best_t - width_fraction * vrange)
    high = min(vmax, best_t + width_fraction * vrange)
    return np.unique(np.linspace(low, high, n_points))


def compute_metrics(sig_mask, bkg_mask):
    tp = int(sig_mask.sum())
    fn = len(sig_mask) - tp
    fp = int(bkg_mask.sum())
    tn = len(bkg_mask) - fp
    n = len(sig_mask) + len(bkg_mask)
    return {
        "TP": tp, "FN": fn, "FP": fp, "TN": tn,
        "accuracy": (tp + tn) / n,
        "signal_efficiency": tp / len(sig_mask),
        "background_efficiency": fp / len(bkg_mask),
        "background_rejection": 1 - fp / len(bkg_mask),
    }


def search_best_rectangular_cut(signal_df, background_df, chosen_features, threshold_dict):
    pre_sig, pre_bkg = {}, {}
    for f in chosen_features:
        xs, xb = signal_df[f].to_numpy(), background_df[f].to_numpy()
        ts = threshold_dict[f]
        pre_sig[f] = {">": [xs > t for t in ts], "<": [xs < t for t in ts]}
        pre_bkg[f] = {">": [xb > t for t in ts], "<": [xb < t for t in ts]}

    f1, f2, f3 = chosen_features
    t1s, t2s, t3s = threshold_dict[f1], threshold_dict[f2], threshold_dict[f3]
    best = None

    for d1, d2, d3 in product([">", "<"], repeat=3):
        for i in range(len(t1s)):
            s1, b1 = pre_sig[f1][d1][i], pre_bkg[f1][d1][i]
            for j in range(len(t2s)):
                s12 = s1 & pre_sig[f2][d2][j]
                b12 = b1 & pre_bkg[f2][d2][j]
                for k in range(len(t3s)):
                    s123 = s12 & pre_sig[f3][d3][k]
                    b123 = b12 & pre_bkg[f3][d3][k]
                    m = compute_metrics(s123, b123)
                    if best is None or m["accuracy"] > best["accuracy"]:
                        best = {
                            "features": chosen_features,
                            "cuts": {f1: (d1, float(t1s[i])),
                                     f2: (d2, float(t2s[j])),
                                     f3: (d3, float(t3s[k]))},
                            **m,
                        }
    return best


coarse_thresholds = {f: get_thresholds(signal[f].to_numpy(), background[f].to_numpy()) for f in top3_features}
best_coarse = search_best_rectangular_cut(signal, background, top3_features, coarse_thresholds)

refined_thresholds = {
    f: refine_thresholds(signal[f].to_numpy(), background[f].to_numpy(), best_coarse["cuts"][f][1])
    for f in top3_features
}
best_final = search_best_rectangular_cut(signal, background, top3_features, refined_thresholds)

print(pd.DataFrame({
    "Feature": top3_features,
    "Direction": [best_final["cuts"][f][0] for f in top3_features],
    "Threshold": [best_final["cuts"][f][1] for f in top3_features],
}).to_string(index=False))
print(f"\naccuracy = {best_final['accuracy']:.4f}, "
      f"sig_eff = {best_final['signal_efficiency']:.4f}, "
      f"bkg_eff = {best_final['background_efficiency']:.4f}")


def apply_rectangular_selection(df, cut_dict):
    mask = np.ones(len(df), dtype=bool)
    for f, (d, t) in cut_dict.items():
        mask &= (df[f].to_numpy() > t) if d == ">" else (df[f].to_numpy() < t)
    return mask


sig_sel = apply_rectangular_selection(signal, best_final["cuts"])
bkg_sel = apply_rectangular_selection(background, best_final["cuts"])
print(f"\nsignal passing: {int(sig_sel.sum())}, background passing: {int(bkg_sel.sum())}")

In [ ]:
for f in top3_features:
    print(f, "sig:", signal[f].min(), signal[f].max(),
             " bkg:", background[f].min(), background[f].max())

In [ ]:
for f, (d, thr) in best_final["cuts"].items():
    col_sig = (signal[f] < thr) if d == "<" else (signal[f] > thr)
    col_bkg = (background[f] < thr) if d == "<" else (background[f] > thr)
    print(f"{f} {d} {thr:.4f}  sig pass {col_sig.mean():.3f}  bkg pass {col_bkg.mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, f in zip(axes, top3_features):
    all_vals = np.concatenate([signal[f].to_numpy(), background[f].to_numpy()])
    bins = np.linspace(all_vals.min(), all_vals.max(), 60)
    ax.hist(background[f], bins=bins, alpha=0.6, density=True, label="Background")
    ax.hist(signal[f], bins=bins, alpha=0.6, density=True, label="Signal")
    d, thr = best_final["cuts"][f]
    ax.axvline(thr, linestyle="--")
    ax.set_title(f"{f} ({d} {thr:.2f})")
    ax.legend()

plt.tight_layout()
plt.savefig("plots/top3_features_with_cuts.png", dpi=300, bbox_inches="tight")
plt.show()